## Importation des Bibliothèques

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration pour l'affichage des graphiques
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

--- 
## PARTIE A - CHARGER ET EXPLORER

### Exercice 1 - Premier contact

In [2]:
# Chargement du fichier csv
chemin_fichier = 'D:\\Programme Akieni\\Programme Akieni\\DataScience\\projet_sante_akieni\\S13\\datasource\\monde_2023_academy.csv'
monde_2023 = pd.read_csv(chemin_fichier)

# Dimensions du dataset avec extraction des elements du tuple
print(f"La dimensions du DataFrame est de : {monde_2023.shape[0]} lignes et {monde_2023.shape[1]} colonnes.\n")

# Aperçu des 5 premières lignes
display(monde_2023.head())

# Types de chaque colonne
print("\nTypes des colonnes :")
print(monde_2023.dtypes)


La dimensions du DataFrame est de : 190 lignes et 7 colonnes.



,country,iso3,continent,year,pop,lifeExp,gdpPercap
0,Brunei,BRN,Asia,2023,458949,75.327,32962.906453
1,United States,USA,Americas,2023,336806231,79.304,81236.427600
2,Dominican Republic,DOM,Americas,2023,11331265,73.720,10717.627671
3,Greece,GRC,Europe,2023,10407351,81.857,22888.275089
4,Burkina Faso,BFA,Africa,2023,23025776,61.092,882.689810



Types des colonnes :
country          str
iso3             str
continent        str
year           int64
pop              str
lifeExp      float64
gdpPercap    float64
dtype: object


In [3]:
# Verification des les lignes où la population totale ('pop') contient une valeur non numérique
monde_2023[pd.to_numeric(monde_2023['pop'], errors='coerce').isna()]

,country,iso3,continent,year,pop,lifeExp,gdpPercap
12,Sierra Leone,SLE,Africa,2023,"8,460,512",NaN,450.307527
18,Mauritania,MRT,Africa,2023,"5,022,441",68.484,2081.174684
71,Benin,BEN,Africa,2023,"14,111,034",60.774,1394.177399
92,Israel,ISR,Asia,2023,"9,849,000",82.408,51771.905341
106,Spain,ESP,Europe,2023,"48,352,528",83.670,32691.045906
131,Ukraine,UKR,Europe,2023,"37,732,836",73.422,4737.439332


**Réponse à la question :**  
La colonne `pop` (population totale) a un type `object` (chaîne de caractères) au lieu d'un type numérique (`int64` ou `float64`). Cela s'explique par la présence de séparateurs de milliers sous forme de virgules (ex. `'8,460,512'`) dans certaines lignes du fichier brut, ce qui empêche Pandas de la reconnaître nativement comme nombre.

### Exercice 2 - Un résumé pour le comité

In [4]:
# Nombre de valeurs renseignées dans chaque colonne
display(monde_2023.count())

# Résumé statistique global des colonnes numériques
display(monde_2023.describe())

country      190
iso3         190
continent    190
year         190
pop          190
lifeExp      185
gdpPercap    185
dtype: int64

,year,lifeExp,gdpPercap
count,190.0,185.000000,185.000000
mean,2023.0,73.247200,17804.797715
std,0.0,7.062549,23957.069605
min,2023.0,54.462000,193.007146
25%,2023.0,67.689000,2477.978455
50%,2023.0,73.720000,7820.103638
75%,2023.0,78.580000,22888.275089
max,2023.0,85.511000,128678.189943


**Réponse à la question :**  
- **Moyenne :** environ **17 804,80 $**
- **Médiane :** environ **7 820,10 $**

La moyenne est plus de deux fois supérieure à la médiane. Cela signifie que **quelques pays très riches font fortement augmenter la moyenne**.
La distribution du PIB par habitant est donc **déséquilibrée vers les grandes valeurs**.
Ainsi, **la médiane représente mieux le PIB par habitant d'un pays typique** que la moyenne.

--- 
## PARTIE B - NETTOYER LES DONNÉES

### Exercice 3 - Des données incomplètes

In [5]:
# Quantification des valeurs manquantes en nombre et en pourcentage
nombre_manquants = monde_2023.isnull().sum()
pourcentage_manquants = (nombre_manquants / len(monde_2023)) * 100

df_valeurs_manquantes = pd.DataFrame({'Valeurs manquantes': nombre_manquants, 'Pourcentage (%)': pourcentage_manquants})
display(df_valeurs_manquantes)

# Traitement : les lignes avec des valeurs manquantes sur lifeExp ou gdpPercap représentent ~2.63% du total.
# Etant donné ce faible pourcentage, on peut soit les supprimer, soit les imputer.
# Pour une analyse macroéconomique rigoureuse, supprimer les lignes incomplètes est acceptable,
# mais veillons à documenter l'opération.

,Valeurs manquantes,Pourcentage (%)
country,0,0.000000
iso3,0,0.000000
continent,0,0.000000
year,0,0.000000
pop,0,0.000000
lifeExp,5,2.631579
gdpPercap,5,2.631579


### Traitement des valeurs manquantes

Les valeurs manquantes dans les colonnes `lifeExp` ou `gdpPercap` représentent environ **2,63 % des observations**.
Comme ce pourcentage est faible, deux solutions sont possibles :

- supprimer les lignes concernées ;
- remplacer les valeurs manquantes par des valeurs estimées.

Pour cette analyse, nous choisissons de **supprimer les lignes incomplètes** afin de travailler uniquement avec des données complètes et fiables.

Cette opération est documentée pour garder une trace du nettoyage effectué.

In [6]:
# Suppression des valeurs manquantes
monde_2023 = monde_2023.dropna(subset=['lifeExp', 'gdpPercap'])

# Verification de l'existance des valeurs manquantes
monde_2023[['lifeExp', 'gdpPercap']].isna().sum()

lifeExp      0
gdpPercap    0
dtype: int64

### Exercice 4 - Un doublon dans l'export

In [7]:
print(f"Nombre de lignes avant suppression des doublons : {len(monde_2023)}")
nombre_doublons = monde_2023.duplicated().sum()
print(f"Nombre de doublons détectés : {nombre_doublons}")

Nombre de lignes avant suppression des doublons : 180
Nombre de doublons détectés : 5


In [8]:
# Suppression des doublons
monde_2023_propre = monde_2023.drop_duplicates().copy()
print(f"Nombre de lignes après suppression : {len(monde_2023_propre)}")

Nombre de lignes après suppression : 175


### Exercice 5 - Un pays introuvable

In [9]:
# Recherche par code ISO 'COG'
display(monde_2023_propre[monde_2023_propre['iso3'] == 'COG'])

# Nettoyage des noms de pays : suppression des espaces et mise en forme
monde_2023_propre['country'] = monde_2023_propre['country'].str.strip().str.title()

# Vérification après correction
display(monde_2023_propre[monde_2023_propre['iso3'] == 'COG'])

,country,iso3,continent,year,pop,lifeExp,gdpPercap
47,congo,COG,Africa,2023,6182885,65.772,2477.978455


,country,iso3,continent,year,pop,lifeExp,gdpPercap
47,Congo,COG,Africa,2023,6182885,65.772,2477.978455


### Exercice 6 - Une colonne numérique qui n'en est pas une

In [10]:
# Suppression des virgules de formatage des milliers et conversion
monde_2023_propre['pop'] = monde_2023_propre['pop'].astype(str).str.replace(',', '').astype(float)

# Vérification
print("Type de la colonne pop après conversion :", monde_2023_propre['pop'].dtype)
display(monde_2023_propre[['country', 'pop']].head())

Type de la colonne pop après conversion : float64


,country,pop
0,Brunei,458949.0
1,United States,336806231.0
2,Dominican Republic,11331265.0
3,Greece,10407351.0
4,Burkina Faso,23025776.0


--- 
## PARTIE C - SÉLECTIONNER ET FILTRER

### Exercice 7 - Trois demandes d'un collègue

In [11]:
# a) Extrait contenant uniquement pays, continent et espérance de vie
extrait_a = monde_2023_propre[['country', 'continent', 'lifeExp']]
print("--- 7.a) Extrait (Pays, Continent, Espérance de vie)")
display(extrait_a.head())

# b) Aperçu des 5 premiers pays
print("--- 7.b1) 5 1ers pays : toutes colonnes")
display(monde_2023_propre.head())

print("--- 7.b2) 5 premiers pays : nom et population")
display(monde_2023_propre[['country', 'pop']].head())

# c) Liste des pays de plus de 100 millions d'habitants (descending)
pays_plus_100m = monde_2023_propre[monde_2023_propre['pop'] > 100_000_000].sort_values(by='pop', ascending=False)
print(f"--- 7.c) Pays de plus de 100 millions d'habitants ({len(pays_plus_100m)} pays)")
display(pays_plus_100m[['country', 'pop', 'continent']].sort_values(by='pop', ascending=False))

--- 7.a) Extrait (Pays, Continent, Espérance de vie)


,country,continent,lifeExp
0,Brunei,Asia,75.327
1,United States,Americas,79.304
2,Dominican Republic,Americas,73.720
3,Greece,Europe,81.857
4,Burkina Faso,Africa,61.092


--- 7.b1) 5 1ers pays : toutes colonnes


,country,iso3,continent,year,pop,lifeExp,gdpPercap
0,Brunei,BRN,Asia,2023,458949.0,75.327,32962.906453
1,United States,USA,Americas,2023,336806231.0,79.304,81236.427600
2,Dominican Republic,DOM,Americas,2023,11331265.0,73.720,10717.627671
3,Greece,GRC,Europe,2023,10407351.0,81.857,22888.275089
4,Burkina Faso,BFA,Africa,2023,23025776.0,61.092,882.689810


--- 7.b2) 5 premiers pays : nom et population


,country,pop
0,Brunei,458949.0
1,United States,336806231.0
2,Dominican Republic,11331265.0
3,Greece,10407351.0
4,Burkina Faso,23025776.0


--- 7.c) Pays de plus de 100 millions d'habitants (14 pays)


,country,pop,continent
135,China,1.410710e+09,Asia
1,United States,3.368062e+08,Americas
170,Indonesia,2.811901e+08,Asia
61,Pakistan,2.475045e+08,Asia
94,Nigeria,2.278829e+08,Africa
128,Brazil,2.111407e+08,Americas
157,Russia,1.438261e+08,Europe
63,Mexico,1.297398e+08,Americas
102,Ethiopia,1.286917e+08,Africa
11,Japan,1.245166e+08,Asia


--- 
## PARTIE D - MOYENNE ET MÉDIANE À L'ÉPREUVE DES FAITS

### Exercice 8 - La richesse "typique" d'un pays africain

In [12]:
# Isolation des pays africains
afrique_df = monde_2023_propre[monde_2023_propre['continent'] == 'Africa']

# Calcul de la moyenne et de la médiane du PIB par habitant en Afrique
richesse_moyenne_afrique = afrique_df['gdpPercap'].mean()
richesse_mediane_afrique = afrique_df['gdpPercap'].median()

print(f"Richesse moyenne en Afrique : {round(richesse_moyenne_afrique, 2)} $")
print(f"Richesse médiane en Afrique : {round(richesse_mediane_afrique, 2)} $")

# Top 5 des PIB/hab en Afrique
print("\nTop 5 des pays africains avec le PIB par habitant le plus élevé :")
display(afrique_df.sort_values(by='gdpPercap', ascending=False)[['country', 'gdpPercap']].head())

Richesse moyenne en Afrique : 2818.31 $
Richesse médiane en Afrique : 1631.58 $

Top 5 des pays africains avec le PIB par habitant le plus élevé :


,country,gdpPercap
34,Seychelles,17879.239655
118,Mauritius,11530.182094
124,Gabon,8256.690765
112,Botswana,7820.103638
150,Libya,6911.316617


**Analyse :**  
La moyenne (~2 786.94 \$) est près de 70% supérieure à la médiane (~1 631.58 \$). Cette asymétrie s'explique par la présence d'économies insulaires ou pétrolières hautement performantes comme les Seychelles, Maurice, le Gabon ou le Botswana, qui tirent la moyenne arithmétique vers le haut, alors que la moitié des pays africains se situent sous la barre des 1 632 \$.

--- 
## PARTIE E - CE QUE LA DISPERSION RÉVÈLE

### Exercice 9 - Deux continents, deux réalités ?

In [13]:
# Comparaison de l'espérance de vie : Afrique vs Europe
comparaison_stats = (
    monde_2023_propre[monde_2023_propre['continent'].isin(['Africa', 'Europe'])]
    .groupby('continent')['lifeExp']
    .agg(
        Espérance_de_vie_Moyenne='mean',
        Écart_type='std',
        Min='min',
        Max='max'
    )
)

display(comparaison_stats)

,Espérance_de_vie_Moyenne,Écart_type,Min,Max
continent,,,,
Africa,65.574860,5.189278,54.462,76.508
Europe,79.817795,3.485239,71.198,84.041


### Exercice 10 - Le cas Congo

In [14]:
# Comparaison entre le Congo Brazza et le Congo Kinshasa
congos = monde_2023_propre[monde_2023_propre['iso3'].isin(['COG', 'COD'])]
display(congos[['country', 'iso3', 'pop', 'lifeExp', 'gdpPercap']])

,country,iso3,pop,lifeExp,gdpPercap
47,Congo,COG,6182885.0,65.772,2477.978455
178,Democratic Republic Of Congo,COD,105789731.0,61.895,627.502182


--- 
## PARTIE F - LE BIAIS DU POIDS DÉMOGRAPHIQUE

### Exercice 11 - Chaque pays compte-t-il pour un ? Ou chaque personne ?

In [15]:
# Nettoyage des lignes valides pour l'espérance de vie et la population
monde_valide = monde_2023_propre.dropna(subset=['lifeExp', 'pop']).copy()

# 1. Moyenne non pondérée (chaque pays = 1 vote)
moyenne_non_ponderee = monde_valide['lifeExp'].mean()

# 2. Moyenne pondérée par la population (chaque habitant = 1 vote)
moyenne_ponderee = np.average(monde_valide['lifeExp'], weights=monde_valide['pop'])

print(f"Espérance de vie moyenne non pondérée (par pays) : {round(moyenne_non_ponderee, 2)} ans")
print(f"Espérance de vie moyenne pondérée (par habitant) : {round(moyenne_ponderee, 2)} ans")

Espérance de vie moyenne non pondérée (par pays) : 73.16 ans
Espérance de vie moyenne pondérée (par habitant) : 73.73 ans


--- 
## PARTIE G - CORRELATION : RICHESSE ET LONGÉVITÉ

### Exercice 12 - La richesse d'un pays est-elle liée à la longévité de ses habitants ?

In [16]:
# Corrélation PIB/hab vs Espérance de vie
correlation_richesse_vie = monde_valide['gdpPercap'].corr(monde_valide['lifeExp'])
print(f"Corrélation : {round(correlation_richesse_vie, 4)}")

# Top 5 des pays riches (PIB > 20k$) avec la plus faible espérance de vie
pays_riches_faible_vie = (
    monde_valide[monde_valide['gdpPercap'] > 20000]
    .sort_values('lifeExp')
    .head(5)
)

print("\nTop 5 des pays (PIB > 20k$) avec la plus faible espérance de vie :")
display(pays_riches_faible_vie[['country', 'continent', 'gdpPercap', 'lifeExp']])

Corrélation : 0.6946

Top 5 des pays (PIB > 20k$) avec la plus faible espérance de vie :


,country,continent,gdpPercap,lifeExp
175,Guyana,Americas,20313.718421,70.180
134,Saint Kitts And Nevis,Americas,23034.199733,72.145
176,Trinidad And Tobago,Americas,20577.505678,73.490
146,Bahamas,Americas,35896.505107,74.552
0,Brunei,Asia,32962.906453,75.327


--- 
## POUR CONCLURE

### Exercice 13 - Restitution écrite

**Synthèse rédigée :**
L'analyse exploratoire du jeu de données mondial 2023 révèle de fortes disparités géo-économiques et démographiques à travers les 184 pays étudiés. En examinant la richesse par habitant, on constate que la **médiane** mondiale s'établit à 7 820,10 \$, tandis que la **moyenne** atteint 17 804,80 \$, illustrant une asymétrie marquée par de fortes **valeurs aberrantes** économiques. Concernant la longévité, l'**écart-type** de l'espérance de vie en Afrique (5,19 ans) est nettement supérieur à celui observé en Europe (3,49 ans), témoignant d'une plus grande hétérogénéité des conditions sanitaires et de développement sur le continent africain. Enfin, la forte **corrélation** positive de 0,69 calculée entre la richesse et l'espérance de vie démontre que le développement économique va de pair avec la longévité, bien que des facteurs sociaux et sanitaires structurels jouent également un rôle déterminant.